# Exploratory Analysis of CA1 Miniscope Calcium Imaging During Fear Conditioning

Self-directed training project using public mouse miniscope data from DANDI:000718. The goal is methodological: NWB access, calcium-trace inspection, event alignment, behavior alignment, and PCA.

This notebook analyzes one animal and one fear-conditioning session. Results are exploratory and descriptive.


## Dataset

- DANDI:000718, published version `0.260825.1902`
- Subject: `Ca-EEG3-4`
- Session: `FC`
- Brain region: CA1
- One-photon miniature calcium imaging
- 783 processed ROIs
- Behavioral annotations include motion, freezing intervals, and three foot shocks

The released NWB file already contains processed optical-physiology outputs. This notebook does not claim to perform the original motion correction, segmentation, or source extraction.


## 1. Setup and NWB access


In [ ]:
!pip install -q dandi pynwb h5py remfile numpy pandas scipy matplotlib scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py, remfile
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO
from scipy.stats import zscore
from sklearn.decomposition import PCA


In [ ]:
DANDISET_ID = "000718"
VERSION = "0.260825.1902"
subject_id = "Ca-EEG3-4"
session_id = "FC"
nwb_path = f"sub-{subject_id}/sub-{subject_id}_ses-{session_id}_image+ophys.nwb"

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(DANDISET_ID, version_id=VERSION)
    asset = dandiset.get_asset_by_path(nwb_path)
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)

remote_file = remfile.File(s3_url)
h5_file = h5py.File(remote_file, "r")
io = NWBHDF5IO(file=h5_file, load_namespaces=True)
nwb = io.read()

print(nwb.session_id, nwb.subject.subject_id)
print(list(nwb.processing.keys()))


## 2. Calcium signals and population visualization


In [ ]:
ophys = nwb.processing["ophys"]
fluorescence = ophys.data_interfaces["Fluorescence"]
denoised_series = fluorescence.roi_response_series["Denoised"]
deconv_series = fluorescence.roi_response_series["Deconvolved"]

calcium = np.asarray(denoised_series.data[:])
deconv = np.asarray(deconv_series.data[:])
timestamps = np.asarray(denoised_series.timestamps[:])
fs = 1 / np.median(np.diff(timestamps))
calcium_z = zscore(calcium, axis=0, nan_policy="omit")

print("Calcium shape:", calcium.shape)
print("Sampling rate:", fs)


In [ ]:
neurons = [20, 150, 300, 500, 700]
plt.figure(figsize=(14, 8))
for offset, neuron in enumerate(neurons):
    trace = calcium_z[:, neuron]
    plt.plot(timestamps, trace + offset*5, linewidth=1, label=f"ROI {neuron}")
plt.xlabel("Time (s)")
plt.ylabel("Z-scored calcium activity (offset)")
plt.title("Example CA1 calcium traces during fear conditioning")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 8))
plt.imshow(calcium_z.T, aspect="auto", origin="lower", extent=[timestamps[0], timestamps[-1], 0, calcium_z.shape[1]], vmin=-2, vmax=4)
plt.colorbar(label="Z-scored calcium activity")
plt.xlabel("Time (s)")
plt.ylabel("ROI")
plt.title("CA1 population calcium activity")
plt.tight_layout()
plt.show()


## 3. Foot-shock aligned activity


In [ ]:
shock_df = nwb.stimulus["ShockStimuli"].to_dataframe()
shock_starts = shock_df["start_time"].to_numpy()
shock_stops = shock_df["stop_time"].to_numpy()
print(shock_df)
population_activity = np.nanmean(calcium_z, axis=1)
active_fraction = np.mean(deconv > 0, axis=1)


In [ ]:
pre_time, post_time = 10, 20
relative_time = np.arange(-pre_time, post_time, 1/fs)
def align_trace(trace):
    return np.array([np.interp(t0 + relative_time, timestamps, trace) for t0 in shock_starts])
shock_aligned_calcium = align_trace(population_activity)
shock_aligned_active = align_trace(active_fraction)
plt.figure(figsize=(11, 6))
for i in range(len(shock_starts)):
    plt.plot(relative_time, shock_aligned_calcium[i], alpha=.5, label=f"Shock {i+1}")
plt.plot(relative_time, shock_aligned_calcium.mean(axis=0), linewidth=3, label="Mean")
plt.axvspan(0, 2, alpha=.2)
plt.axvline(0, linestyle="--")
plt.xlabel("Time relative to shock onset (s)")
plt.ylabel("Mean CA1 activity (z-score)")
plt.title("CA1 population activity aligned to foot shock")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
pre_mask = (relative_time >= -10) & (relative_time < 0)
shock_mask = (relative_time >= 0) & (relative_time < 2)
post_mask = (relative_time >= 2) & (relative_time < 10)
def epoch_summary(aligned):
    return pd.DataFrame({"Shock": np.arange(1, len(shock_starts)+1), "Pre": aligned[:, pre_mask].mean(axis=1), "During": aligned[:, shock_mask].mean(axis=1), "Post": aligned[:, post_mask].mean(axis=1)})
print("Denoised population activity")
display(epoch_summary(shock_aligned_calcium))
print("Deconvolved active fraction")
display(epoch_summary(shock_aligned_active))


**Interpretation.** The three shocks did not show a uniform population-wide response. The analysis is therefore treated descriptively, not as evidence for a consistent shock effect.


## 4. Freezing versus non-freezing activity


In [ ]:
behavior = nwb.processing["behavior"]
freezing_df = behavior.data_interfaces["FreezingIntervals"].to_dataframe()
freezing_mask = np.zeros(len(timestamps), dtype=bool)
for _, row in freezing_df.iterrows():
    freezing_mask |= (timestamps >= row["start_time"]) & (timestamps <= row["stop_time"])
print("Freezing intervals:", len(freezing_df))
print("Fraction of recording spent freezing:", freezing_mask.mean())


In [ ]:
freezing_summary = pd.DataFrame({"Metric": ["Mean population activity", "Fraction of active ROIs"], "Freezing": [population_activity[freezing_mask].mean(), active_fraction[freezing_mask].mean()], "Non-freezing": [population_activity[~freezing_mask].mean(), active_fraction[~freezing_mask].mean()]})
freezing_summary["Difference"] = freezing_summary["Freezing"] - freezing_summary["Non-freezing"]
display(freezing_summary)


In [ ]:
roi_difference_z = calcium_z[freezing_mask].mean(axis=0) - calcium_z[~freezing_mask].mean(axis=0)
roi_active_difference = (deconv[freezing_mask] > 0).mean(axis=0) - (deconv[~freezing_mask] > 0).mean(axis=0)
print("ROIs higher during freezing:", np.sum(roi_difference_z > 0))
print("ROIs lower during freezing:", np.sum(roi_difference_z < 0))
print("ROIs with higher event fraction during freezing:", np.sum(roi_active_difference > 0))
print("ROIs with lower event fraction during freezing:", np.sum(roi_active_difference < 0))
plt.figure(figsize=(8,5))
plt.hist(roi_difference_z, bins=40)
plt.axvline(0, linestyle="--")
plt.xlabel("Mean z-scored activity difference (freezing - non-freezing)")
plt.ylabel("Number of ROIs")
plt.title("ROI-wise standardized activity differences during freezing")
plt.tight_layout()
plt.show()


**Interpretation.** Freezing occupied about 6% of the session. Global activity differences were small, while individual ROIs showed heterogeneous increases and decreases.


## 5. PCA of CA1 population dynamics


In [ ]:
pca = PCA(n_components=10)
population_pca = pca.fit_transform(calcium_z)
print("PCA shape:", population_pca.shape)
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {var*100:.2f}%")
print(f"PC1 + PC2: {pca.explained_variance_ratio_[:2].sum()*100:.2f}%")
print(f"First 10 PCs: {pca.explained_variance_ratio_.sum()*100:.2f}%")


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(np.arange(1,11), pca.explained_variance_ratio_*100, marker="o")
plt.xlabel("Principal component")
plt.ylabel("Explained variance (%)")
plt.title("Variance explained by CA1 population PCs")
plt.xticks(range(1,11))
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(population_pca[~freezing_mask,0], population_pca[~freezing_mask,1], s=10, alpha=.25, label="Non-freezing")
plt.scatter(population_pca[freezing_mask,0], population_pca[freezing_mask,1], s=18, alpha=.7, label="Freezing")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("CA1 population state during freezing and non-freezing")
plt.legend()
plt.tight_layout()
plt.show()


## Summary

The CA1 recording contained heterogeneous activity across 783 segmented ROIs. Peri-shock responses were not uniform across the three shock events. Freezing occupied about 6% of the analyzed session and was associated with only small changes in global population metrics, while ROI-level responses were mixed. PCA indicated distributed population dynamics, with PC1 and PC2 together explaining about 9.7% of variance and substantial overlap between freezing and non-freezing states.


## Limitations

- One animal and one recording session.
- Calcium traces and ROI segmentation were already processed in the public release.
- Deconvolved activity is inferred event-like activity, not directly recorded spikes.
- Adjacent time points are autocorrelated and are not independent biological replicates.
- Freezing comparisons are exploratory and descriptive.
- No causal or population-level generalization is claimed.


## Data source

DANDI:000718, published version `0.260825.1902`

https://dandiarchive.org/dandiset/000718/0.260825.1902
